**Notas**

- Notebook basado en el curso de Andrew Ng, *Deep Learning Specialization* (DeepLearning.AI / Coursera).
- Modificado y editado con la ayuda de Claude.
- Revisado y editado por el autor final: Alvaro Riascos.
- Todos los errores son responsabilidad exclusiva del autor.

**Versión taller**

Completa el código en cada celda marcada con `# TU CÓDIGO EMPIEZA AQUÍ` y `# TU CÓDIGO TERMINA AQUÍ`. Los comentarios que están justo antes de cada bloque indican cuántas líneas se esperan y qué variables debes definir. Después de cada ejercicio, ejecuta la celda de prueba que sigue y compara con el **Resultado esperado**. No cambies los nombres de las funciones ni de sus argumentos: las pruebas automáticas dependen de ellos.

# Clasificación de datos planares con una capa oculta

¡Es hora de construir tu primera red neuronal, que tendrá una capa oculta! Notarás una gran diferencia entre este modelo y el que implementaste antes con regresión logística.

Al terminar este notebook podrás:

- Implementar una red neuronal de clasificación binaria con una sola capa oculta
- Usar unidades con una función de activación no lineal, como tanh
- Calcular la pérdida de entropía cruzada (*cross-entropy*)
- Implementar la propagación hacia adelante y hacia atrás

## Contenido
- [1 - Paquetes](#1)
- [2 - Cargar los datos](#2)
    - [Ejercicio 1](#ex-1)
- [3 - Regresión logística simple](#3)
- [4 - Modelo de red neuronal](#4)
    - [4.1 - Definir la estructura de la red neuronal](#4-1)
        - [Ejercicio 2 - layer_sizes](#ex-2)
    - [4.2 - Inicializar los parámetros del modelo](#4-2)
        - [Ejercicio 3 - initialize_parameters](#ex-3)
    - [4.3 - El ciclo](#4-3)
        - [Ejercicio 4 - forward_propagation](#ex-4)
    - [4.4 - Calcular el costo](#4-4)
        - [Ejercicio 5 - compute_cost](#ex-5)
    - [4.5 - Implementar la retropropagación](#4-5)
        - [Ejercicio 6 - backward_propagation](#ex-6)
    - [4.6 - Actualizar los parámetros](#4-6)
        - [Ejercicio 7 - update_parameters](#ex-7)
    - [4.7 - Integración](#4-7)
        - [Ejercicio 8 - nn_model](#ex-8)
- [5 - Probar el modelo](#5)
    - [5.1 - Predecir](#5-1)
        - [Ejercicio 9 - predict](#ex-9)
    - [5.2 - Probar el modelo con los datos planares](#5-2)
- [6 - Ajustar el tamaño de la capa oculta (opcional)](#6)
- [7 - Desempeño en otros conjuntos de datos](#7)

<a name='1'></a>
# 1 - Paquetes

Primero importa todos los paquetes que necesitarás en este notebook.

- [numpy](https://www.numpy.org/) es el paquete fundamental para computación científica en Python.
- [sklearn](http://scikit-learn.org/stable/) ofrece herramientas simples y eficientes para minería y análisis de datos.
- [matplotlib](http://matplotlib.org) es una librería para hacer gráficas en Python.
- testCases ofrece ejemplos de prueba para evaluar si tus funciones son correctas.
- planar_utils ofrece varias funciones útiles que se usan en este notebook.

In [ ]:
# Importar paquetes
import numpy as np
import copy
import matplotlib.pyplot as plt
from testCases_v2 import *
from public_tests import *
import sklearn
import sklearn.datasets
import sklearn.linear_model
from planar_utils import plot_decision_boundary, sigmoid, load_planar_dataset, load_extra_datasets

%matplotlib inline

%load_ext autoreload
%autoreload 2

<a name='2'></a>
# 2 - Cargar los datos


In [ ]:
X, Y = load_planar_dataset()

Visualiza los datos con matplotlib. Los datos parecen una "flor" con algunos puntos rojos (etiqueta y=0) y otros azules (y=1). Tu objetivo es construir un modelo que se ajuste a estos datos. En otras palabras, queremos que el clasificador defina regiones rojas o azules.

In [ ]:
# Visualizar los datos:
plt.scatter(X[0, :], X[1, :], c=Y, s=40, cmap=plt.cm.Spectral);

Tienes:
- un arreglo de numpy (matriz) X que contiene las variables (x1, x2)
- un arreglo de numpy (vector) Y que contiene las etiquetas (rojo: 0, azul: 1).

Primero, entiende mejor cómo son tus datos.

<a name='ex-1'></a>
### Ejercicio 1

¿Cuántos ejemplos de entrenamiento tienes? Además, ¿cuál es la forma (`shape`) de las variables `X` y `Y`?

**Pista**: ¿cómo se obtiene la forma de un arreglo de numpy? [(ayuda)](https://numpy.org/doc/stable/reference/generated/numpy.ndarray.shape.html)

In [ ]:
# (≈ 3 líneas de código)
# shape_X = ...
# shape_Y = ...
# tamaño del conjunto de entrenamiento
# m = ...
# TU CÓDIGO EMPIEZA AQUÍ


# TU CÓDIGO TERMINA AQUÍ

print ('La forma de X es: ' + str(shape_X))
print ('La forma de Y es: ' + str(shape_Y))
print ('¡Tengo m = %d ejemplos de entrenamiento!' % (m))

**Resultado esperado**:

<table style="width:20%">
  <tr>
    <td> forma de X </td>
    <td> (2, 400) </td>
  </tr>
  <tr>
    <td>forma de Y</td>
    <td>(1, 400) </td>
    </tr>
    <tr>
    <td>m</td>
    <td> 400 </td>
  </tr>
</table>

<a name='3'></a>
## 3 - Regresión logística simple

Antes de construir una red neuronal completa, veamos qué tan bien funciona la regresión logística en este problema. Puedes usar las funciones de sklearn para esto. Ejecuta el código de abajo para entrenar un clasificador de regresión logística sobre los datos.

In [ ]:
# Entrenar el clasificador de regresión logística
# (fijamos scoring y l1_ratios explícitamente y silenciamos las FutureWarning de scikit-learn
#  sobre cambios futuros en sus valores por defecto; no afectan el resultado)
import warnings
with warnings.catch_warnings():
    warnings.simplefilter("ignore", FutureWarning)
    clf = sklearn.linear_model.LogisticRegressionCV(scoring='accuracy', l1_ratios=(0,))
    clf.fit(X.T, np.ravel(Y))

Ahora puedes graficar la frontera de decisión del modelo. Ejecuta el código de abajo.

**Nota sobre el cálculo de la precisión (*accuracy*)**: la versión original de este notebook calculaba la precisión con `float(np.dot(Y, predicciones) + ...)`. Como `Y` tiene forma (1, m) y las predicciones forman un vector de m elementos, `np.dot` devuelve un **arreglo**, no un escalar, y desde NumPy 2.x `float()` solo acepta arreglos de 0 dimensiones. Por eso aquí usamos `np.mean(predicciones == Y)`, que es directamente la proporción de puntos bien clasificados.

In [ ]:
# Graficar la frontera de decisión de la regresión logística
plot_decision_boundary(lambda x: clf.predict(x), X, Y)
plt.title("Regresión logística")

# Imprimir la precisión
LR_predictions = clf.predict(X.T)
accuracy = np.mean(LR_predictions == Y) * 100
print ('Precisión de la regresión logística: %d %% (porcentaje de puntos bien clasificados)' % accuracy)

**Resultado esperado**:

<table style="width:20%">
  <tr>
    <td>Precisión</td>
    <td> 47% </td>
  </tr>

</table>


**Interpretación**: los datos no son linealmente separables, así que la regresión logística no funciona bien. Ojalá una red neuronal lo haga mejor. ¡Probemos!

<a name='4'></a>
## 4 - Modelo de red neuronal

La regresión logística no funcionó bien con los datos de la "flor". Ahora vas a entrenar una red neuronal con una sola capa oculta y ver cómo resuelve el mismo problema.

**El modelo**:
<img src="images/classification_kiank.png" style="width:600px;height:300px;">

**Matemáticamente**:

Para un ejemplo $x^{(i)}$:
$$z^{[1] (i)} =  W^{[1]} x^{(i)} + b^{[1]}\tag{1}$$
$$a^{[1] (i)} = \tanh(z^{[1] (i)})\tag{2}$$
$$z^{[2] (i)} = W^{[2]} a^{[1] (i)} + b^{[2]}\tag{3}$$
$$\hat{y}^{(i)} = a^{[2] (i)} = \sigma(z^{ [2] (i)})\tag{4}$$
$$y^{(i)}_{predicción} = \begin{cases} 1 & \mbox{si } a^{[2](i)} > 0.5 \\ 0 & \mbox{en otro caso } \end{cases}\tag{5}$$

Dadas las predicciones sobre todos los ejemplos, también puedes calcular el costo $J$ así:
$$J = - \frac{1}{m} \sum\limits_{i = 1}^{m} \large\left(\small y^{(i)}\log\left(a^{[2] (i)}\right) + (1-y^{(i)})\log\left(1- a^{[2] (i)}\right)  \large  \right) \small \tag{6}$$

**Recordatorio**: la metodología general para construir una red neuronal es:
1. Definir la estructura de la red (número de unidades de entrada, número de unidades ocultas, etc.).
2. Inicializar los parámetros del modelo.
3. Iterar:
    - Implementar la propagación hacia adelante
    - Calcular la pérdida
    - Implementar la propagación hacia atrás para obtener los gradientes
    - Actualizar los parámetros (descenso por gradiente)

En la práctica, se suelen construir funciones auxiliares para los pasos 1-3 y luego integrarlas en una sola función llamada `nn_model()`. Una vez construida `nn_model()` y aprendidos los parámetros, puedes hacer predicciones sobre datos nuevos.

<a name='4-1'></a>
### 4.1 - Definir la estructura de la red neuronal ####

**Nota**: en este ejercicio la función se prueba con datos cuya salida tiene dimensión n_y = 2, pero la red que vamos a construir para los datos de la "flor" tiene salida de dimensión n_y = 1.

<a name='ex-2'></a>
### Ejercicio 2 - layer_sizes

Define tres variables:
- n_x: el tamaño de la capa de entrada
- n_h: el tamaño de la capa oculta (**fíjalo en 4, `n_h = 4`, pero solo para este Ejercicio 2**)
- n_y: el tamaño de la capa de salida

**Pista**: usa las formas de X y Y para encontrar n_x y n_y. Además, fija el tamaño de la capa oculta en 4.

In [ ]:
# FUNCIÓN CALIFICADA: layer_sizes

def layer_sizes(X, Y):
    """
    Argumentos:
    X -- datos de entrada de forma (tamaño de entrada, número de ejemplos)
    Y -- etiquetas de forma (tamaño de salida, número de ejemplos)

    Retorna:
    n_x -- el tamaño de la capa de entrada
    n_h -- el tamaño de la capa oculta
    n_y -- el tamaño de la capa de salida
    """
    #(≈ 3 líneas de código)
    # n_x = ...
    # n_h = ...
    # n_y = ...
    # TU CÓDIGO EMPIEZA AQUÍ
    
    
    # TU CÓDIGO TERMINA AQUÍ
    return (n_x, n_h, n_y)

In [ ]:
t_X, t_Y = layer_sizes_test_case()
(n_x, n_h, n_y) = layer_sizes(t_X, t_Y)
print("El tamaño de la capa de entrada es: n_x = " + str(n_x))
print("El tamaño de la capa oculta es: n_h = " + str(n_h))
print("El tamaño de la capa de salida es: n_y = " + str(n_y))

layer_sizes_test(layer_sizes)

***Resultado esperado***
```
El tamaño de la capa de entrada es: n_x = 5
El tamaño de la capa oculta es: n_h = 4
El tamaño de la capa de salida es: n_y = 2
All tests passed!
```

<a name='4-2'></a>
### 4.2 - Inicializar los parámetros del modelo ####

<a name='ex-3'></a>
### Ejercicio 3 - initialize_parameters

Implementa la función `initialize_parameters()`.

**Instrucciones**:
- Asegúrate de que los tamaños de los parámetros sean correctos. Si es necesario, revisa la figura de la red neuronal de arriba.
- Inicializa las matrices de pesos con valores aleatorios.
    - Usa: `np.random.randn(a,b) * 0.01` para inicializar aleatoriamente una matriz de forma (a,b).
- Inicializa los vectores de sesgo en ceros.
    - Usa: `np.zeros((a,b))` para inicializar con ceros una matriz de forma (a,b).

In [ ]:
# FUNCIÓN CALIFICADA: initialize_parameters

def initialize_parameters(n_x, n_h, n_y):
    """
    Argumentos:
    n_x -- tamaño de la capa de entrada
    n_h -- tamaño de la capa oculta
    n_y -- tamaño de la capa de salida

    Retorna:
    params -- diccionario de python con los parámetros:
                    W1 -- matriz de pesos de forma (n_h, n_x)
                    b1 -- vector de sesgos de forma (n_h, 1)
                    W2 -- matriz de pesos de forma (n_y, n_h)
                    b2 -- vector de sesgos de forma (n_y, 1)
    """
    #(≈ 4 líneas de código)
    # W1 = ...
    # b1 = ...
    # W2 = ...
    # b2 = ...
    # TU CÓDIGO EMPIEZA AQUÍ
    
    
    # TU CÓDIGO TERMINA AQUÍ

    parameters = {"W1": W1,
                  "b1": b1,
                  "W2": W2,
                  "b2": b2}

    return parameters

In [ ]:
np.random.seed(2)
n_x, n_h, n_y = initialize_parameters_test_case()
parameters = initialize_parameters(n_x, n_h, n_y)

print("W1 = " + str(parameters["W1"]))
print("b1 = " + str(parameters["b1"]))
print("W2 = " + str(parameters["W2"]))
print("b2 = " + str(parameters["b2"]))

initialize_parameters_test(initialize_parameters)

**Resultado esperado**
```
W1 = [[-0.00416758 -0.00056267]
 [-0.02136196  0.01640271]
 [-0.01793436 -0.00841747]
 [ 0.00502881 -0.01245288]]
b1 = [[0.]
 [0.]
 [0.]
 [0.]]
W2 = [[-0.01057952 -0.00909008  0.00551454  0.02292208]]
b2 = [[0.]]
All tests passed!
```

<a name='4-3'></a>
### 4.3 - El ciclo

<a name='ex-4'></a>
### Ejercicio 4 - forward_propagation

Implementa `forward_propagation()` usando las siguientes ecuaciones:

$$Z^{[1]} =  W^{[1]} X + b^{[1]}\tag{1}$$
$$A^{[1]} = \tanh(Z^{[1]})\tag{2}$$
$$Z^{[2]} = W^{[2]} A^{[1]} + b^{[2]}\tag{3}$$
$$\hat{Y} = A^{[2]} = \sigma(Z^{[2]})\tag{4}$$


**Instrucciones**:

- Revisa la representación matemática del clasificador en la figura de arriba.
- Usa la función `sigmoid()`. Ya está importada en este notebook.
- Usa la función `np.tanh()`. Es parte de la librería numpy.
- Sigue estos pasos:
    1. Obtén cada parámetro del diccionario "parameters" (que es la salida de `initialize_parameters()`) usando `parameters[".."]`.
    2. Implementa la propagación hacia adelante. Calcula $Z^{[1]}, A^{[1]}, Z^{[2]}$ y $A^{[2]}$ (el vector con todas las predicciones sobre todos los ejemplos del conjunto de entrenamiento).
- Los valores que se necesitan en la retropropagación se guardan en "cache". El cache se pasa como entrada a la función de retropropagación.

In [ ]:
# FUNCIÓN CALIFICADA: forward_propagation

def forward_propagation(X, parameters):
    """
    Argumentos:
    X -- datos de entrada de tamaño (n_x, m)
    parameters -- diccionario de python con los parámetros (salida de la función de inicialización)

    Retorna:
    A2 -- la salida sigmoide de la segunda activación
    cache -- un diccionario con "Z1", "A1", "Z2" y "A2"
    """
    # Obtener cada parámetro del diccionario "parameters"
    #(≈ 4 líneas de código)
    # W1 = ...
    # b1 = ...
    # W2 = ...
    # b2 = ...
    # TU CÓDIGO EMPIEZA AQUÍ
    
    
    # TU CÓDIGO TERMINA AQUÍ

    # Implementar la propagación hacia adelante para calcular A2 (probabilidades)
    # (≈ 4 líneas de código)
    # Z1 = ...
    # A1 = ...
    # Z2 = ...
    # A2 = ...
    # TU CÓDIGO EMPIEZA AQUÍ
    
    
    # TU CÓDIGO TERMINA AQUÍ

    assert(A2.shape == (1, X.shape[1]))

    cache = {"Z1": Z1,
             "A1": A1,
             "Z2": Z2,
             "A2": A2}

    return A2, cache

In [ ]:
t_X, parameters = forward_propagation_test_case()
A2, cache = forward_propagation(t_X, parameters)
print("A2 = " + str(A2))

forward_propagation_test(forward_propagation)

***Resultado esperado***
```
A2 = [[0.21292656 0.21274673 0.21295976]]
All tests passed!
```

<a name='4-4'></a>
### 4.4 - Calcular el costo

Ahora que calculaste $A^{[2]}$ (en la variable de Python "`A2`"), que contiene $a^{[2](i)}$ para todos los ejemplos, puedes calcular la función de costo así:

$$J = - \frac{1}{m} \sum\limits_{i = 1}^{m} \large{(} \small y^{(i)}\log\left(a^{[2] (i)}\right) + (1-y^{(i)})\log\left(1- a^{[2] (i)}\right) \large{)} \small\tag{13}$$

<a name='ex-5'></a>
### Ejercicio 5 - compute_cost

Implementa `compute_cost()` para calcular el valor del costo $J$.

**Instrucciones**:
- Hay muchas formas de implementar la pérdida de entropía cruzada. Esta es una forma de implementar una parte de la ecuación sin ciclos for:
$- \sum\limits_{i=1}^{m}  y^{(i)}\log(a^{[2](i)})$:
```python
logprobs = np.multiply(np.log(A2),Y)
cost = - np.sum(logprobs)
```

- Úsala para construir la expresión completa de la función de costo.

**Notas**:

- Puedes usar `np.multiply()` y luego `np.sum()`, o directamente `np.dot()`.
- Si usas `np.multiply` seguido de `np.sum`, el resultado será de tipo `float`; si usas `np.dot`, el resultado será un arreglo 2D de numpy.
- Puedes usar `np.squeeze()` para eliminar dimensiones redundantes (en el caso de un solo número, queda reducido a un arreglo de dimensión cero).
- También puedes convertir el arreglo a tipo `float` usando `float()`. Ojo: desde NumPy 2.x, `float()` solo funciona sobre arreglos de dimensión cero, por eso primero se aplica `np.squeeze()`.

In [ ]:
# FUNCIÓN CALIFICADA: compute_cost

def compute_cost(A2, Y):
    """
    Calcula el costo de entropía cruzada dado en la ecuación (13)

    Argumentos:
    A2 -- la salida sigmoide de la segunda activación, de forma (1, número de ejemplos)
    Y -- vector de etiquetas "verdaderas" de forma (1, número de ejemplos)

    Retorna:
    cost -- costo de entropía cruzada dado en la ecuación (13)

    """

    m = Y.shape[1] # número de ejemplos

    # Calcular el costo de entropía cruzada
    # (≈ 2 líneas de código)
    # logprobs = ...
    # cost = ...
    # TU CÓDIGO EMPIEZA AQUÍ
    
    
    # TU CÓDIGO TERMINA AQUÍ

    cost = float(np.squeeze(cost))  # garantiza que cost tenga la dimensión esperada.
                                    # Ej.: convierte [[17]] en 17

    return cost

In [ ]:
A2, t_Y = compute_cost_test_case()
cost = compute_cost(A2, t_Y)
print("cost = " + str(cost))

compute_cost_test(compute_cost)

***Resultado esperado***
```
cost = 0.6930587610394646
All tests passed!
```


<a name='4-5'></a>
### 4.5 - Implementar la retropropagación

Usando el cache calculado durante la propagación hacia adelante, ahora puedes implementar la propagación hacia atrás.

<a name='ex-6'></a>
### Ejercicio 6 - backward_propagation

Implementa la función `backward_propagation()`.

**Instrucciones**:
La retropropagación suele ser la parte más difícil (la más matemática) del *deep learning*. Para ayudarte, aquí está de nuevo la diapositiva de la clase sobre retropropagación. Usa las seis ecuaciones de la derecha, ya que estás construyendo una implementación vectorizada.

<img src="images/grad_summary.png" style="width:600px;height:300px;">
<caption><center><font color='purple'><b>Figura 1</b>: Retropropagación. Usa las seis ecuaciones de la derecha.</font></center></caption>

<!--
$\frac{\partial \mathcal{J} }{ \partial z_{2}^{(i)} } = \frac{1}{m} (a^{[2](i)} - y^{(i)})$

$\frac{\partial \mathcal{J} }{ \partial W_2 } = \frac{\partial \mathcal{J} }{ \partial z_{2}^{(i)} } a^{[1] (i) T} $

$\frac{\partial \mathcal{J} }{ \partial b_2 } = \sum_i{\frac{\partial \mathcal{J} }{ \partial z_{2}^{(i)}}}$

$\frac{\partial \mathcal{J} }{ \partial z_{1}^{(i)} } =  W_2^T \frac{\partial \mathcal{J} }{ \partial z_{2}^{(i)} } * ( 1 - a^{[1] (i) 2}) $

$\frac{\partial \mathcal{J} }{ \partial W_1 } = \frac{\partial \mathcal{J} }{ \partial z_{1}^{(i)} }  X^T $

$\frac{\partial \mathcal{J} _i }{ \partial b_1 } = \sum_i{\frac{\partial \mathcal{J} }{ \partial z_{1}^{(i)}}}$

- Note that $*$ denotes elementwise multiplication.
- The notation you will use is common in deep learning coding:
    - dW1 = $\frac{\partial \mathcal{J} }{ \partial W_1 }$
    - db1 = $\frac{\partial \mathcal{J} }{ \partial b_1 }$
    - dW2 = $\frac{\partial \mathcal{J} }{ \partial W_2 }$
    - db2 = $\frac{\partial \mathcal{J} }{ \partial b_2 }$

!-->

- Consejos:
    - Para calcular dZ1 necesitas calcular $g^{[1]'}(Z^{[1]})$. Como $g^{[1]}(.)$ es la función de activación tanh, si $a = g^{[1]}(z)$ entonces $g^{[1]'}(z) = 1-a^2$. Así que puedes calcular
    $g^{[1]'}(Z^{[1]})$ usando `(1 - np.power(A1, 2))`.

In [ ]:
# FUNCIÓN CALIFICADA: backward_propagation

def backward_propagation(parameters, cache, X, Y):
    """
    Implementa la propagación hacia atrás siguiendo las instrucciones de arriba.

    Argumentos:
    parameters -- diccionario de python con los parámetros
    cache -- un diccionario con "Z1", "A1", "Z2" y "A2".
    X -- datos de entrada de forma (2, número de ejemplos)
    Y -- vector de etiquetas "verdaderas" de forma (1, número de ejemplos)

    Retorna:
    grads -- diccionario de python con los gradientes con respecto a los distintos parámetros
    """
    m = X.shape[1]

    # Primero, obtener W1 y W2 del diccionario "parameters".
    #(≈ 2 líneas de código)
    # W1 = ...
    # W2 = ...
    # TU CÓDIGO EMPIEZA AQUÍ
    
    
    # TU CÓDIGO TERMINA AQUÍ

    # Obtener también A1 y A2 del diccionario "cache".
    #(≈ 2 líneas de código)
    # A1 = ...
    # A2 = ...
    # TU CÓDIGO EMPIEZA AQUÍ
    
    
    # TU CÓDIGO TERMINA AQUÍ

    # Propagación hacia atrás: calcular dW1, db1, dW2, db2.
    #(≈ 6 líneas de código, correspondientes a las 6 ecuaciones de la diapositiva)
    # dZ2 = ...
    # dW2 = ...
    # db2 = ...
    # dZ1 = ...
    # dW1 = ...
    # db1 = ...
    # TU CÓDIGO EMPIEZA AQUÍ
    
    
    # TU CÓDIGO TERMINA AQUÍ

    grads = {"dW1": dW1,
             "db1": db1,
             "dW2": dW2,
             "db2": db2}

    return grads

In [ ]:
parameters, cache, t_X, t_Y = backward_propagation_test_case()

grads = backward_propagation(parameters, cache, t_X, t_Y)
print ("dW1 = "+ str(grads["dW1"]))
print ("db1 = "+ str(grads["db1"]))
print ("dW2 = "+ str(grads["dW2"]))
print ("db2 = "+ str(grads["db2"]))

backward_propagation_test(backward_propagation)

***Resultado esperado***
```
dW1 = [[ 0.00301023 -0.00747267]
 [ 0.00257968 -0.00641288]
 [-0.00156892  0.003893  ]
 [-0.00652037  0.01618243]]
db1 = [[ 0.00176201]
 [ 0.00150995]
 [-0.00091736]
 [-0.00381422]]
dW2 = [[ 0.00078841  0.01765429 -0.00084166 -0.01022527]]
db2 = [[-0.16655712]]
All tests passed!
```

<a name='4-6'></a>
### 4.6 - Actualizar los parámetros

<a name='ex-7'></a>
### Ejercicio 7 - update_parameters

Implementa la regla de actualización. Usa descenso por gradiente. Debes usar (dW1, db1, dW2, db2) para actualizar (W1, b1, W2, b2).

**Regla general del descenso por gradiente**: $\theta = \theta - \alpha \frac{\partial J }{ \partial \theta }$, donde $\alpha$ es la tasa de aprendizaje y $\theta$ representa un parámetro.

<img src="images/sgd.gif" style="width:400;height:400;"> <img src="images/sgd_bad.gif" style="width:400;height:400;">
<caption><center><font color='purple'><b>Figura 2</b>: El algoritmo de descenso por gradiente con una buena tasa de aprendizaje (converge) y con una mala tasa de aprendizaje (diverge). Imágenes cortesía de Adam Harley.</font></center></caption>

**Pista**

- Usa `copy.deepcopy(...)` al copiar listas o diccionarios que se pasan como parámetros a funciones. Así se evita que la función modifique los parámetros de entrada.

**Nota**: si escribes la actualización como `W1 = W1 - learning_rate*dW1`, no hace falta `copy.deepcopy`, porque esa expresión crea un arreglo **nuevo** en vez de modificar el original. En cambio, una actualización "en el lugar" como `W1 -= learning_rate*dW1` sí modificaría el arreglo guardado en el diccionario `parameters` que recibió la función; en ese caso sí habría que copiarlo primero.

In [ ]:
# FUNCIÓN CALIFICADA: update_parameters

def update_parameters(parameters, grads, learning_rate = 1.2):
    """
    Actualiza los parámetros usando la regla de descenso por gradiente dada arriba

    Argumentos:
    parameters -- diccionario de python con los parámetros
    grads -- diccionario de python con los gradientes

    Retorna:
    parameters -- diccionario de python con los parámetros actualizados
    """
    # Obtener cada parámetro del diccionario "parameters" (ver la nota de arriba sobre copy.deepcopy)
    #(≈ 4 líneas de código)
    # W1 = ...
    # b1 = ...
    # W2 = ...
    # b2 = ...
    # TU CÓDIGO EMPIEZA AQUÍ
    
    
    # TU CÓDIGO TERMINA AQUÍ

    # Obtener cada gradiente del diccionario "grads"
    #(≈ 4 líneas de código)
    # dW1 = ...
    # db1 = ...
    # dW2 = ...
    # db2 = ...
    # TU CÓDIGO EMPIEZA AQUÍ
    
    
    # TU CÓDIGO TERMINA AQUÍ

    # Regla de actualización para cada parámetro
    #(≈ 4 líneas de código)
    # W1 = ...
    # b1 = ...
    # W2 = ...
    # b2 = ...
    # TU CÓDIGO EMPIEZA AQUÍ
    
    
    # TU CÓDIGO TERMINA AQUÍ

    parameters = {"W1": W1,
                  "b1": b1,
                  "W2": W2,
                  "b2": b2}

    return parameters

In [ ]:
parameters, grads = update_parameters_test_case()
parameters = update_parameters(parameters, grads)

print("W1 = " + str(parameters["W1"]))
print("b1 = " + str(parameters["b1"]))
print("W2 = " + str(parameters["W2"]))
print("b2 = " + str(parameters["b2"]))

update_parameters_test(update_parameters)

***Resultado esperado***
```
W1 = [[-0.00643025  0.01936718]
 [-0.02410458  0.03978052]
 [-0.01653973 -0.02096177]
 [ 0.01046864 -0.05990141]]
b1 = [[-1.02420756e-06]
 [ 1.27373948e-05]
 [ 8.32996807e-07]
 [-3.20136836e-06]]
W2 = [[-0.01041081 -0.04463285  0.01758031  0.04747113]]
b2 = [[0.00010457]]
All tests passed!
```

<a name='4-7'></a>
### 4.7 - Integración

Integra tus funciones en `nn_model()`.

<a name='ex-8'></a>
### Ejercicio 8 - nn_model

Construye tu modelo de red neuronal en `nn_model()`.

**Instrucciones**: el modelo de red neuronal debe usar las funciones anteriores en el orden correcto.

In [ ]:
# FUNCIÓN CALIFICADA: nn_model

def nn_model(X, Y, n_h, num_iterations = 10000, print_cost=False):
    """
    Argumentos:
    X -- datos de forma (2, número de ejemplos)
    Y -- etiquetas de forma (1, número de ejemplos)
    n_h -- tamaño de la capa oculta
    num_iterations -- número de iteraciones del ciclo de descenso por gradiente
    print_cost -- si es True, imprime el costo cada 1000 iteraciones

    Retorna:
    parameters -- parámetros aprendidos por el modelo. Se pueden usar para predecir.
    """

    np.random.seed(3)
    n_x = layer_sizes(X, Y)[0]
    n_y = layer_sizes(X, Y)[2]

    # Inicializar los parámetros
    #(≈ 1 línea de código)
    # parameters = ...
    # TU CÓDIGO EMPIEZA AQUÍ
    
    
    # TU CÓDIGO TERMINA AQUÍ

    # Ciclo (descenso por gradiente)

    for i in range(0, num_iterations):

        #(≈ 4 líneas de código)
        # Propagación hacia adelante. Entradas: "X, parameters". Salidas: "A2, cache".
        # A2, cache = ...

        # Función de costo. Entradas: "A2, Y". Salidas: "cost".
        # cost = ...

        # Retropropagación. Entradas: "parameters, cache, X, Y". Salidas: "grads".
        # grads = ...

        # Actualización de parámetros por descenso por gradiente. Entradas: "parameters, grads". Salidas: "parameters".
        # parameters = ...

        # TU CÓDIGO EMPIEZA AQUÍ
        
        
        # TU CÓDIGO TERMINA AQUÍ

        # Imprimir el costo cada 1000 iteraciones
        if print_cost and i % 1000 == 0:
            print ("Costo después de la iteración %i: %f" %(i, cost))

    return parameters

In [ ]:
nn_model_test(nn_model)

***Resultado esperado***
```
Costo después de la iteración 0: 0.693086
Costo después de la iteración 1000: 0.000220
Costo después de la iteración 2000: 0.000108
...
Costo después de la iteración 8000: 0.000027
Costo después de la iteración 9000: 0.000024
W1 = [[ 0.71392202  1.31281102]
 [-0.76411243 -1.41967065]
 [-0.75040545 -1.38857337]
 [ 0.56495575  1.04857776]]
b1 = [[-0.0073536 ]
 [ 0.01534663]
 [ 0.01262938]
 [ 0.00218135]]
W2 = [[ 2.82545815 -3.3063945  -3.16116615  1.8549574 ]]
b2 = [[0.00393452]]
All tests passed!
```

<a name='5'></a>
## 5 - Probar el modelo

<a name='5-1'></a>
### 5.1 - Predecir

<a name='ex-9'></a>
### Ejercicio 9 - predict

Haz predicciones con tu modelo construyendo `predict()`.
Usa la propagación hacia adelante para predecir los resultados.

**Recordatorio**: predicciones = $y_{predicción} = \mathbb 1 \text{{activación > 0.5}} = \begin{cases}
      1 & \text{si}\ activación > 0.5 \\
      0 & \text{en otro caso}
    \end{cases}$

Por ejemplo, si quieres convertir las entradas de una matriz X en 0 y 1 según un umbral, harías: ```X_new = (X > threshold)```

In [ ]:
# FUNCIÓN CALIFICADA: predict

def predict(parameters, X):
    """
    Usando los parámetros aprendidos, predice una clase para cada ejemplo de X

    Argumentos:
    parameters -- diccionario de python con los parámetros
    X -- datos de entrada de tamaño (n_x, m)

    Retorna:
    predictions -- vector de predicciones del modelo (rojo: 0 / azul: 1)
    """

    # Calcula las probabilidades con la propagación hacia adelante y clasifica en 0/1 usando 0.5 como umbral.
    #(≈ 2 líneas de código)
    # A2, cache = ...
    # predictions = ...
    # TU CÓDIGO EMPIEZA AQUÍ
    
    
    # TU CÓDIGO TERMINA AQUÍ

    return predictions

In [ ]:
parameters, t_X = predict_test_case()

predictions = predict(parameters, t_X)
print("Predicciones: " + str(predictions))

predict_test(predict)

***Resultado esperado***
```
Predicciones: [[ True False  True]]
All tests passed!
```

<a name='5-2'></a>
### 5.2 - Probar el modelo con los datos planares

Es hora de ejecutar el modelo y ver cómo funciona con los datos planares. ¡Ejecuta el siguiente código para probar tu modelo con una sola capa oculta de $n_h$ unidades!

In [ ]:
# Construir un modelo con una capa oculta de n_h unidades
parameters = nn_model(X, Y, n_h = 4, num_iterations = 10000, print_cost=True)

# Graficar la frontera de decisión
plot_decision_boundary(lambda x: predict(parameters, x.T), X, Y)
plt.title("Frontera de decisión con capa oculta de tamaño " + str(4))

In [ ]:
# Imprimir la precisión
predictions = predict(parameters, X)
accuracy = np.mean(predictions == Y) * 100
print ('Precisión: %d%%' % accuracy)

**Resultado esperado**:

<table style="width:30%">
  <tr>
    <td><b>Precisión</b></td>
    <td> 90% </td>
  </tr>
</table>

La precisión es mucho mayor que la de la regresión logística. ¡El modelo aprendió los patrones de los pétalos de la flor! A diferencia de la regresión logística, las redes neuronales pueden aprender fronteras de decisión muy no lineales.

### ¡Felicitaciones por terminar este notebook!

Un resumen rápido de lo que acabas de lograr:

- Construiste una red neuronal completa de clasificación binaria con una capa oculta
- Usaste una unidad no lineal
- Calculaste la pérdida de entropía cruzada
- Implementaste la propagación hacia adelante y hacia atrás
- Viste el impacto de cambiar el tamaño de la capa oculta, incluido el sobreajuste

¡Creaste una red neuronal que puede aprender patrones! Excelente trabajo. Abajo hay algunos ejercicios opcionales para probar otros tamaños de capa oculta y otros conjuntos de datos.

<a name='6'></a>
## 6 - Ajustar el tamaño de la capa oculta (opcional)

Ejecuta el siguiente código (puede tardar 1-2 minutos). Luego observa los distintos comportamientos del modelo para varios tamaños de capa oculta.

In [ ]:
# Esto puede tardar unos 2 minutos

plt.figure(figsize=(16, 24))
hidden_layer_sizes = [1, 2, 3, 4, 5, 12]

# puedes probar con otros tamaños de capa oculta, por ejemplo:
# hidden_layer_sizes = [1, 2, 3, 4, 5, 20, 50]

for i, n_h in enumerate(hidden_layer_sizes):
    plt.subplot((len(hidden_layer_sizes) + 1) // 2, 2, i+1)
    plt.title('Capa oculta de tamaño %d' % n_h)
    parameters = nn_model(X, Y, n_h, num_iterations = 5000)
    plot_decision_boundary(lambda x: predict(parameters, x.T), X, Y)
    predictions = predict(parameters, X)
    accuracy = np.mean(predictions == Y) * 100
    print ("Precisión con {} unidades ocultas: {} %".format(n_h, accuracy))

**Interpretación**:
- Los modelos más grandes (con más unidades ocultas) se ajustan mejor al conjunto de entrenamiento, hasta que eventualmente los modelos más grandes sobreajustan los datos.
- El mejor tamaño de capa oculta parece estar alrededor de n_h = 5. En efecto, un valor cercano a este parece ajustarse bien a los datos sin incurrir en un sobreajuste notable.
- Más adelante veremos la regularización, que permite usar modelos muy grandes (como n_h = 50) sin mucho sobreajuste.

**Algunas preguntas opcionales que puedes explorar**:
- ¿Qué pasa si cambias la activación tanh por una activación sigmoide o ReLU?
- Juega con la tasa de aprendizaje (learning_rate). ¿Qué pasa?
- ¿Qué pasa si cambiamos el conjunto de datos? (¡Ver la parte 7 abajo!)

<a name='7'></a>
## 7 - Desempeño en otros conjuntos de datos

Si quieres, puedes volver a ejecutar el notebook completo (excepto la parte de los datos) con cada uno de los siguientes conjuntos de datos. La primera celda carga y grafica el conjunto elegido; la segunda entrena la red neuronal sobre él y muestra la frontera de decisión y la precisión.

In [ ]:
# Conjuntos de datos
noisy_circles, noisy_moons, blobs, gaussian_quantiles, no_structure = load_extra_datasets()

datasets = {"noisy_circles": noisy_circles,
            "noisy_moons": noisy_moons,
            "blobs": blobs,
            "gaussian_quantiles": gaussian_quantiles}

### EMPIEZA EL CÓDIGO AQUÍ ### (elige el conjunto de datos)
dataset = "noisy_moons"
### TERMINA EL CÓDIGO AQUÍ ###

X, Y = datasets[dataset]
X, Y = X.T, Y.reshape(1, Y.shape[0])

# convertir blobs en un problema binario
if dataset == "blobs":
    Y = Y%2

# Visualizar los datos
plt.scatter(X[0, :], X[1, :], c=Y, s=40, cmap=plt.cm.Spectral);

In [ ]:
# Entrenar la red neuronal sobre el conjunto de datos elegido
parameters = nn_model(X, Y, n_h = 4, num_iterations = 10000, print_cost=False)

# Graficar la frontera de decisión
plot_decision_boundary(lambda x: predict(parameters, x.T), X, Y)
plt.title("Frontera de decisión (" + dataset + ") con capa oculta de tamaño " + str(4))

# Imprimir la precisión
predictions = predict(parameters, X)
accuracy = np.mean(predictions == Y) * 100
print ('Precisión en ' + dataset + ': %d%%' % accuracy)

**Referencias**:

- http://scs.ryerson.ca/~aharley/neural-networks/
- http://cs231n.github.io/neural-networks-case-study/